# data-cleaning_006

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (126).ipynb`

| Field | Value |
|---|---|
| Section | `data_cleaning` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 33 |
| Detected functions | is_float, convert, remove_pps_outliers, location_chart |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
dataset = pd.read_csv('Pune_House_Data.csv')
dataset

In [ ]:
dataset = dataset.drop(['area_type', 'availability', 'society', 'balcony'], axis = 1)
dataset.head()

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset['bath'] = dataset['bath'].fillna(dataset.bath.mean())
dataset

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset['size'].unique()

In [ ]:
dataset['size'] = dataset['size'].apply(lambda x: int(x.split(' ')[0]) if isinstance(x, str) else x)

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset

In [ ]:
dataset['size'].unique()

In [ ]:
dataset['size'] = dataset['size'].fillna(dataset['size'].mean())
dataset

In [ ]:
dataset['size'].unique()

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset.total_sqft.unique()

In [ ]:
def is_float(x):
    try:
        float(x)
    except:
        return False
    return True

In [ ]:
dataset[~dataset['total_sqft'].apply(is_float)]

In [ ]:
def convert(x):
    token = x.split('-')
    if len(token) == 2:
        return (float(token[0]) + float(token[1]))/2
    try:
        return float(x)
    except:
        return None

In [ ]:
dataset['total_sqft'] = dataset['total_sqft'].apply(convert)
dataset

In [ ]:
dataset.total_sqft.unique()

In [ ]:
dataset

In [ ]:
dataset['price_per_sqft'] = dataset['price']*100000/dataset['total_sqft']
dataset

In [ ]:
len(dataset.site_location.unique())

In [ ]:
dataset.shape

In [ ]:
dataset.price_per_sqft.describe()

In [ ]:
dataset['price_per_sqft'] = dataset['price_per_sqft'].astype(int)
dataset = dataset.price_per_sqft.fillna()

In [ ]:
dataset.price_per_sqft.describe()

In [ ]:
dataset.total_sqft.describe()

In [ ]:
# Fill NaN values with an empty string and then strip
dataset.site_location = dataset.site_location.fillna('').apply(lambda x: x.strip())


location_status = dataset.groupby('site_location')['site_location'].agg('count')
location_status

In [ ]:
def remove_pps_outliers(dataset):
    df_out = pd.DataFrame()  # Initialize df_out before the loop
    for key, subdf in dataset.groupby('site_location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        reduced_df = subdf[(subdf.price_per_sqft > (m - st)) & (subdf.price_per_sqft < (m + st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out

# Usage
dataset = remove_pps_outliers(dataset)
dataset.shape


In [ ]:
import matplotlib.pyplot as plt

def location_chart(dataset, site_location):
    bhk2 = dataset[(dataset.site_location == site_location) & (dataset.size == 2)]
    bhk3 = dataset[(dataset.site_location == site_location) & (dataset.size == 3)]
    plt.rcParams['figure.figsize'] = (14, 7)
    plt.scatter(bhk2.total_sqft, bhk2.price_per_sqft, color='blue', label='2 BHK', s=50)
    plt.scatter(bhk3.total_sqft, bhk3.price_per_sqft, marker='+', color='red', label='3 BHK', s=50)
    plt.xlabel('Total Square Feet')
    plt.ylabel('Price per Square Feet')
    plt.title(site_location)
    plt.legend()
    plt.show()  # Add this to display the plot

# Example usage
location_chart(dataset, 'Alandi Road')
